# W261 Final Project Phase II - Baseline Modeling
**Team:** Section 3, Group 2

This notebook trains baseline models. We will first do model training and selection based on raw data. Then do the feature engineering. After that, we will use new feature set to go through the same modeling pipeline to do the model training and selection based on the feature engineered data. Eventually, we will do a final result comparison between models trained from raw data vs feature engineered data and get our final winner for baseline modeling part.

Going forward, we will use that model as our baseline model.

Load data from parquet

In [0]:
from pyspark.sql.functions import col, count, when, isnan, avg, sum as spark_sum, lit, desc
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

spark.conf.set("spark.sql.ansi.enabled", "false")

# Load data from parquet
TEAM_DIR = "dbfs:/student-groups/Group_3_2"
df_3m = spark.read.parquet(f"{TEAM_DIR}/otpw_3m.parquet")
df_12m = spark.read.parquet(f"{TEAM_DIR}/otpw_12m.parquet")

print(f"3-Month OTPW:  {df_3m.count():,} rows")
print(f"12-Month OTPW: {df_12m.count():,} rows")


# Data Cleaning & Feature Preparation

We structure our modeling in two phases:
1. **Baseline (raw features):** Train models using the original columns with minimal preprocessing (type casting, null imputation). This establishes a performance floor.
2. **Engineered features:** Apply feature transformations identified during EDA and evaluate improvement over baseline.

## Filter to FM-15 and Remove Cancelled Flights

We filter both datasets to FM-15 records only (87.5% of flights) to ensure consistent hourly weather coverage, and exclude cancelled flights since they have no departure delay to predict.

In [0]:
from pyspark.sql.functions import regexp_replace
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

df_model_3m = df_3m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)
df_model_12m = df_12m.filter(col("REPORT_TYPE") == "FM-15").filter(col("CANCELLED") == 0)

print(f"3-Month (FM-15, non-cancelled):  {df_model_3m.count():,} rows")
print(f"12-Month (FM-15, non-cancelled): {df_model_12m.count():,} rows")

### Clean Weather Columns

Raw weather values contain extraneous characters (e.g., `'s'` suffixes for suspect readings, `'*'` placeholders). We apply regex rules from our Phase I weather EDA to strip these and cast to proper numeric types.

In [0]:
hourly_clean_rules = {
    "HourlyStationPressure": ["s", "", "double"],
    "HourlySeaLevelPressure": ["s", "", "double"],
    "HourlyAltimeterSetting": ["s", "", "double"],
    "HourlyDewPointTemperature": ["[^0-9\\.\\-]", "", "double"],
    "HourlyDryBulbTemperature": ["[^0-9\\.\\-]", "", "double"],
    "HourlyWindSpeed": ["s", "", "double"],
    "HourlyWindGustSpeed": ["s", "", "double"],
    "HourlyVisibility": ["[^0-9\\.]", "", "double"],
    "HourlyRelativeHumidity": ["\\*", "0", "double"],
    "HourlyWindDirection": ["[^0-9]", "", "double"],
    "HourlyPrecipitation": ["[^0-9\\.]", "", "double"],
}

def clean_weather(df):
    for c_name, (regex_str, replace_with, cast_type) in hourly_clean_rules.items():
        if c_name in df.columns:
            df = df.withColumn(c_name, regexp_replace(col(c_name), regex_str, replace_with).cast(cast_type))
    return df

df_model_3m = clean_weather(df_model_3m)
df_model_12m = clean_weather(df_model_12m)
print("Weather columns cleaned and cast to numeric.")

### Select Baseline Features and Impute Nulls

We select only the raw features identified in the EDA with no transformations as this moment. Just type casting and null imputation. Per the NOAA documentation, FM-15 weather nulls are imputed as 0 since they represent "none observed." This gives us a clean baseline to measure raw feature predictive power before applying any feature engineering.

In [0]:
from pyspark.sql.functions import log1p, mean, stddev, when, concat_ws

label_col = "DEP_DEL15"

raw_cols = [
    label_col, "DISTANCE", "CRS_DEP_TIME",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity",
    "OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH"
]

numeric_features = [
    "DISTANCE",
    "HourlyDryBulbTemperature", "HourlyDewPointTemperature",
    "HourlyWindSpeed", "HourlyWindGustSpeed",
    "HourlyVisibility", "HourlyPrecipitation",
    "HourlyRelativeHumidity"
]

categorical_features = ["OP_UNIQUE_CARRIER", "ORIGIN", "DAY_OF_WEEK", "MONTH", "DEP_HOUR"]

def prepare_features_baseline(df):
    df = df.select(raw_cols)
    df = df.withColumn(label_col, col(label_col).cast("int"))
    df = df.filter(col(label_col).isNotNull())

    for c in numeric_features:
        df = df.withColumn(c, col(c).cast("double"))
        df = df.fillna({c: 0.0})

    # Extract hour of day from scheduled departure time
    df = df.withColumn("DEP_HOUR", (col("CRS_DEP_TIME").cast("int") / 100).cast("int"))
    df = df.drop("CRS_DEP_TIME")

    return df

df_model_3m = prepare_features_baseline(df_model_3m)
df_model_12m = prepare_features_baseline(df_model_12m)

print(f"3-Month prepared:  {df_model_3m.count():,} rows x {len(df_model_3m.columns)} columns")
print(f"12-Month prepared: {df_model_12m.count():,} rows x {len(df_model_12m.columns)} columns")
df_model_12m.printSchema()

Following are all features we will go ahead using in our baseline modeling:

-  |-- DEP_DEL15: integer
-  |-- DISTANCE: double
-  |-- CRS_DEP_TIME: string
-  |-- HourlyDryBulbTemperature: double
-  |-- HourlyDewPointTemperature: double
-  |-- HourlyWindSpeed: double 
-  |-- HourlyWindGustSpeed: double
-  |-- HourlyVisibility: double
-  |-- HourlyPrecipitation: double
-  |-- HourlyRelativeHumidity: double
-  |-- OP_UNIQUE_CARRIER: string
-  |-- ORIGIN: string
-  |-- DAY_OF_WEEK: string
-  |-- MONTH: string
-  |-- DEP_HOUR: integer

String features will be handled later in the baseline modeling part.

# Baseline Modeling (Raw Features)

We first train models using the raw features with no transformations. This establishes a performance baseline before applying feature engineering.

### Build Spark ML Pipeline

The pipeline applies `StringIndexer`, `OneHotEncoder` for each categorical feature.

In [0]:
cat_features_to_encode = categorical_features

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_features_to_encode
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in cat_features_to_encode
]

assembler_inputs = numeric_features + [f"{c}_ohe" for c in cat_features_to_encode]
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="skip")

pipeline_stages = indexers + encoders + [assembler]

print(f"Pipeline stages: {len(pipeline_stages)}")
print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(cat_features_to_encode)}): {cat_features_to_encode}")
print(f"Total assembler inputs: {len(assembler_inputs)}")

### Train/Test Split

**3-Month dataset (stepping stone):** We split months 1â€“2 for training and month 3 for testing. This validates the pipeline before scaling to the full 12-month dataset.

**12-Month dataset (primary evaluation):** We use **rolling cross-validation on a quarterly basis**. Standard k-fold CV is inappropriate for time-series data because randomly shuffling temporal data would allow the model to train on future flights and predict past ones, leaking temporal information. Instead, we use a sliding window approach for the first two folds and a cumulative training set for the final fold:

| Fold | Train | Test |
|------|-------|------|
| 1 | Q1 (months 1â€“3) | Q2 (months 4â€“6) |
| 2 | Q2 (months 4â€“6) | Q3 (months 7â€“9) |
| 3 | Q1â€“Q3 (months 1â€“9) | Q4 (months 10â€“12) |

Folds 1â€“2 use a single-quarter sliding window to test how well patterns from one quarter generalize to the next. Fold 3 uses all available training data (Q1â€“Q3) to predict the held-out Q4 test set, simulating real-world deployment.

In [0]:
# 3-Month: simple split for pipeline validation
train_3m = df_model_3m.filter(col("MONTH").cast("int") <= 2)
test_3m = df_model_3m.filter(col("MONTH").cast("int") > 2)

print("3-Month split (stepping stone):")
print(f"  Train (months 1-2): {train_3m.count():,}")
print(f"  Test  (month 3):    {test_3m.count():,}")

### Evaluation Metrics

For airlines, the cost of a **false negative** (predicting on-time when the flight is actually delayed) is significantly higher than a **false positive** (predicting delayed when the flight departs on time):

- **False Negative (missed delay):** The airline fails to prepare: crew scheduling is disrupted last-minute, gate reassignments cascade, passengers miss connections, and a single missed delay can trigger a chain of downstream disruptions across the network. The financial and reputational cost is high.
- **False Positive (false alarm):** The airline pre-allocates standby resources that weren't needed: extra crew on standby, a backup gate reserved, passengers notified of a potential delay. The cost is moderate (some wasted resources) but overall harmless.

This makes **recall** our primary metric for model selection: we want to maximize the fraction of actual delays we correctly identify, even at the cost of some false alarms. Precision is monitored as a secondary metric to ensure the model doesn't generate so many false alarms.

| Metric | Role | Why |
|--------|------|-----|
| **Recall** | Primary | Maximizes detection of actual delays, directly reduces the high-cost false negatives |
| **Precision** | Secondary | Ensures predicted delays are good enough for operational action |
| **AUC-PR** | Summary | Captures the full precision-recall trade-off across thresholds |
| **F1** | Comparison | Mean of precision and recall; useful for comparing models |

### Baseline Modeling Approaches

We use **Logistic Regression** as the baseline. We then compare against tree-based models:

- **Random Forest:** Handles non-linear feature interactions without explicit engineering. Robust to outliers and noisy features.
- **Gradient Boosted Trees:** Sequentially builds trees to correct previous errors, typically achieving higher predictive performance than Random Forest. More prone to overfitting.

From the airline's perspective: if the priority is **maximizing delay detection** (recall), tree-based models typically outperform linear baselines because delays are driven by complex feature combinations.

In [0]:
def evaluate_model(predictions, label_col="DEP_DEL15"):
    evaluator_pr = BinaryClassificationEvaluator(labelCol=label_col, metricName="areaUnderPR")
    evaluator_f1 = MulticlassClassificationEvaluator(labelCol=label_col, metricName="f1")

    tp = predictions.filter((col("prediction") == 1) & (col(label_col) == 1)).count()
    fp = predictions.filter((col("prediction") == 1) & (col(label_col) == 0)).count()
    fn = predictions.filter((col("prediction") == 0) & (col(label_col) == 1)).count()

    precision_1 = round(tp / (tp + fp), 4) if (tp + fp) > 0 else 0.0
    recall_1 = round(tp / (tp + fn), 4) if (tp + fn) > 0 else 0.0
    f1_1 = round(2 * precision_1 * recall_1 / (precision_1 + recall_1), 4) if (precision_1 + recall_1) > 0 else 0.0

    return {
        "AUC-PR": round(evaluator_pr.evaluate(predictions), 4),
        "F1 (weighted)": round(evaluator_f1.evaluate(predictions), 4),
        "Precision (delay)": precision_1,
        "Recall (delay)": recall_1,
        "F1 (delay)": f1_1,
    }

def run_experiment(model, pipeline_stages, train_df, test_df, model_name, dataset_name):
    full_pipeline = Pipeline(stages=pipeline_stages + [model])

    start = time.time()
    fitted = full_pipeline.fit(train_df)
    train_time = round(time.time() - start, 1)

    train_preds = fitted.transform(train_df)
    test_preds = fitted.transform(test_df)

    train_metrics = evaluate_model(train_preds)
    test_metrics = evaluate_model(test_preds)

    print(f"\n{'='*60}")
    print(f"{model_name} on {dataset_name}")
    print(f"Training time: {train_time}s")
    print(f"{'='*60}")
    print(f"{'Metric':<20} {'Train':>10} {'Test':>10}")
    print(f"{'-'*42}")
    for m in train_metrics:
        print(f"{m:<20} {train_metrics[m]:>10.4f} {test_metrics[m]:>10.4f}")

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Train Time (s)": train_time,
        **{f"Train_{k}": v for k, v in train_metrics.items()},
        **{f"Test_{k}": v for k, v in test_metrics.items()},
    }, fitted

## 3-Month Baseline Experiments

We start with the 3-month dataset to validate the pipeline before scaling to the full 12-month cross-validation. This uses raw features with undersampling for class imbalance.

In [0]:
results_3m = []

# --- Undersample majority class (on-time flights) to address class imbalance ---
def undersample_majority(df, label_col="DEP_DEL15", seed=42):
    """Undersample the majority class (label=0) to match the minority class count."""
    major_df = df.filter(col(label_col) == 0)
    minor_df = df.filter(col(label_col) == 1)
    major_count = major_df.count()
    minor_count = minor_df.count()
    undersample_ratio = minor_count / major_count
    major_undersampled = major_df.sample(withReplacement=False, fraction=undersample_ratio, seed=seed)
    return major_undersampled.unionAll(minor_df)

train_3m_balanced = undersample_majority(train_3m)
delayed_pct = train_3m_balanced.filter(col(label_col) == 1).count() / train_3m_balanced.count()
print(f"After undersampling â€” delayed class ratio: {delayed_pct:.2%}")
print(f"Balanced train size: {train_3m_balanced.count():,}")

lr_3m = LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20)
lr_3m_result, lr_3m_fitted = run_experiment(lr_3m, pipeline_stages, train_3m_balanced, test_3m, "Logistic Regression", "3-Month")
results_3m.append(lr_3m_result)

In [0]:
rf_3m = RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42)
rf_3m_result, rf_3m_fitted = run_experiment(rf_3m, pipeline_stages, train_3m_balanced, test_3m, "Random Forest", "3-Month")
results_3m.append(rf_3m_result)

In [0]:
gbt_3m = GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42)
gbt_3m_result, gbt_3m_fitted = run_experiment(gbt_3m, pipeline_stages, train_3m_balanced, test_3m, "Gradient Boosted Trees", "3-Month")
results_3m.append(gbt_3m_result)

### 3-Month Baseline Results Summary

In [0]:
results_3m_df = pd.DataFrame(results_3m)
display(results_3m_df)

## 12-Month Baseline Cross Validation Experiments

We apply the 3-fold quarterly rolling CV described in the Train/Test Split section. The last quarter (Q4) serves as the blind test set â€” it is the test set for fold 3, and the model has never seen Q4 data during folds 1 or 2. Results are reported per fold and averaged across all three folds.

In [0]:
# 3-fold quarterly rolling CV
cv_folds = [
    {"train_start": 1, "train_end": 3,  "test_start": 4,  "test_end": 6,  "label": "Train Q1, Test Q2"},
    {"train_start": 4, "train_end": 6,  "test_start": 7,  "test_end": 9,  "label": "Train Q2, Test Q3"},
    {"train_start": 1, "train_end": 9,  "test_start": 10, "test_end": 12, "label": "Train Q1-Q3, Test Q4"},
]

def rolling_cv(df, model_fn, pipeline_stages, folds, label_col="DEP_DEL15"):
    """Rolling cross-validation with quarterly folds and majority undersampling."""
    fold_results = []
    fitted_models = []
    
    for i, fold in enumerate(folds, 1):
        train_fold = df.filter(
            (col("MONTH").cast("int") >= fold["train_start"]) &
            (col("MONTH").cast("int") <= fold["train_end"])
        )
        test_fold = df.filter(
            (col("MONTH").cast("int") >= fold["test_start"]) & 
            (col("MONTH").cast("int") <= fold["test_end"])
        )
        
        test_count = test_fold.count()
        if test_count == 0:
            continue
        
        # Undersample majority class in the training fold
        train_balanced = undersample_majority(train_fold, label_col=label_col, seed=42 + i)
        
        model = model_fn()
        full_pipeline = Pipeline(stages=pipeline_stages + [model])
        
        start = time.time()
        fitted = full_pipeline.fit(train_balanced)
        train_time = round(time.time() - start, 1)
        fitted_models.append(fitted)
        
        test_preds = fitted.transform(test_fold)
        metrics = evaluate_model(test_preds)
        
        fold_results.append({
            "Fold": i, "Split": fold["label"],
            "Train_Size": train_balanced.count(), "Test_Size": test_count,
            "Train_Time_s": train_time, **metrics
        })
        
        print(f"Fold {i}: {fold['label']} | "
              f"Recall={metrics['Recall (delay)']:.4f}, Precision={metrics['Precision (delay)']:.4f}, "
              f"AUC-PR={metrics['AUC-PR']:.4f} ({train_time}s)")
    
    fold_df = pd.DataFrame(fold_results)
    print(f"\n{'='*60}")
    print("Average across folds:")
    for metric in ["AUC-PR", "Recall (delay)", "Precision (delay)", "F1 (delay)"]:
        print(f"  {metric}: {fold_df[metric].mean():.4f} (+/- {fold_df[metric].std():.4f})")
    

    return fold_df, fitted_modelsprint("rolling_cv function defined (3 quarterly folds, with majority undersampling).")


In [0]:
print("=" * 60)
print("Rolling CV: Logistic Regression (3 quarterly folds)")
print("=" * 60)
lr_cv, lr_cv_models = rolling_cv(
    df_model_12m,
    model_fn=lambda: LogisticRegression(featuresCol="features", labelCol=label_col, maxIter=20),
    pipeline_stages=pipeline_stages, folds=cv_folds
)

print("\n" + "=" * 60)
print("Rolling CV: Random Forest (3 quarterly folds)")
print("=" * 60)
rf_cv, rf_cv_models = rolling_cv(
    df_model_12m,
    model_fn=lambda: RandomForestClassifier(featuresCol="features", labelCol=label_col, numTrees=50, maxDepth=10, seed=42),
    pipeline_stages=pipeline_stages, folds=cv_folds
)

print("\n" + "=" * 60)
print("Rolling CV: Gradient Boosted Trees (3 quarterly folds)")
print("=" * 60)
gbt_cv, gbt_cv_models = rolling_cv(
    df_model_12m,
    model_fn=lambda: GBTClassifier(featuresCol="features", labelCol=label_col, maxIter=50, maxDepth=5, seed=42),
    pipeline_stages=pipeline_stages, folds=cv_folds
)

### 12-Month Baseline Cross Validation Results

In [0]:
# CV Summary Table
cv_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "Gradient Boosted Trees"],
    "Avg Recall (delay)": [lr_cv["Recall (delay)"].mean(), rf_cv["Recall (delay)"].mean(), gbt_cv["Recall (delay)"].mean()],
    "Std Recall (delay)": [lr_cv["Recall (delay)"].std(), rf_cv["Recall (delay)"].std(), gbt_cv["Recall (delay)"].std()],
    "Avg Precision (delay)": [lr_cv["Precision (delay)"].mean(), rf_cv["Precision (delay)"].mean(), gbt_cv["Precision (delay)"].mean()],
    "Avg AUC-PR": [lr_cv["AUC-PR"].mean(), rf_cv["AUC-PR"].mean(), gbt_cv["AUC-PR"].mean()],
    "Avg F1 (delay)": [lr_cv["F1 (delay)"].mean(), rf_cv["F1 (delay)"].mean(), gbt_cv["F1 (delay)"].mean()],
})
print("12-Month Rolling CV Summary (3 quarterly folds):")
display(cv_summary)

# Per-fold detail
print("\nPer-fold results:")
all_cv = pd.concat([
    lr_cv.assign(Model="LR"), rf_cv.assign(Model="RF"), gbt_cv.assign(Model="GBT")
])
display(all_cv)

In [0]:
# Recall across quarterly folds â€” visualize model stability
fig, ax = plt.subplots(figsize=(10, 5))
fold_labels = ["Fold 1\nTrain Q1\nTest Q2", "Fold 2\nTrain Q2\nTest Q3", "Fold 3\nTrain Q1-Q3\nTest Q4"]
x = range(len(fold_labels))
width = 0.25

ax.bar([i - width for i in x], lr_cv["Recall (delay)"], width, label="Logistic Regression")
ax.bar(x, rf_cv["Recall (delay)"], width, label="Random Forest")
ax.bar([i + width for i in x], gbt_cv["Recall (delay)"], width, label="Gradient Boosted Trees")
ax.set_xlabel("CV Fold")
ax.set_ylabel("Recall (delay class)")
ax.set_title("Baseline Quarterly Rolling CV: Delay Recall by Fold (12-Month OTPW)")
ax.set_xticks(x)
ax.set_xticklabels(fold_labels)
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Save and Load Baseline Models

Save all fitted baseline pipeline models to our team DBFS directory for reuse in Phase 3 without retraining.

In [0]:
# Save fitted models
model_dir = f"{TEAM_DIR}/models"

lr_3m_fitted.write().overwrite().save(f"{model_dir}/lr_3m")
rf_3m_fitted.write().overwrite().save(f"{model_dir}/rf_3m")
gbt_3m_fitted.write().overwrite().save(f"{model_dir}/gbt_3m")

# Save the fold 3 models (trained on Q1-Q3, the full training set)
lr_cv_models[2].write().overwrite().save(f"{model_dir}/lr_12m")
rf_cv_models[2].write().overwrite().save(f"{model_dir}/rf_12m")
gbt_cv_models[2].write().overwrite().save(f"{model_dir}/gbt_12m")

print(f"All models saved to {model_dir}")

In [0]:
# Load models
from pyspark.ml import PipelineModel

lr_3m_loaded = PipelineModel.load(f"{model_dir}/lr_3m")
rf_3m_loaded = PipelineModel.load(f"{model_dir}/rf_3m")
gbt_3m_loaded = PipelineModel.load(f"{model_dir}/gbt_3m")

lr_12m_loaded = PipelineModel.load(f"{model_dir}/lr_12m")
rf_12m_loaded = PipelineModel.load(f"{model_dir}/rf_12m")
gbt_12m_loaded = PipelineModel.load(f"{model_dir}/gbt_12m")

print("All models loaded.")

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- Train/Test Split (quarterly rolling, 12M dataset) ---
# Fold 1: Train Q1, Test Q2
# Fold 2: Train Q1-Q2, Test Q3
# Final: Train Q1-Q3, Test Q4 (blind)

df_model_12m = df_model_12m.withColumn("MONTH", col("MONTH").cast("int"))

train_df = df_model_12m.filter(col("MONTH") <= 9)   # Q1â€“Q3
test_df  = df_model_12m.filter(col("MONTH") >= 10)  # Q4 blind test

# --- Fit pipeline (transform only, no classifier yet) ---
from pyspark.ml import Pipeline
pipeline_prep = Pipeline(stages=pipeline_stages)  # indexers + encoders + assembler
prep_model = pipeline_prep.fit(train_df)

train_prep = prep_model.transform(train_df)
test_prep  = prep_model.transform(test_df)

# Cache for reuse across models
train_prep.cache()
test_prep.cache()
print(f"Train: {train_prep.count():,} | Test: {test_prep.count():,}")